# Optuna ConvNeXt WSSS — Train, Tune & Evaluate (không DenseCRF)

Notebook này làm theo đúng khung của `test/Optuna_WSSS_Pipeline.ipynb` nhưng:

- Thay 3 backbone ResNet bằng **`convnext_tiny` + `convnext_small`**.
- **Bỏ hoàn toàn DenseCRF** → còn **4 phương pháp**:
  1. `Vanilla CAM`
  2. `CAM + SAM`
  3. `LayerCAM`
  4. `LayerCAM + SAM`
- Kết quả cuối: **2 backbone × 4 phương pháp = 8 cấu hình** trên `Segmentation/val.txt` (1,449 ảnh).

## Quy trình
1. Train classifier ConvNeXt trên `Main/trainval \ Seg/val` (10,418 ảnh).
2. Tune Optuna trên tập cân bằng từ `Seg/train` (mặc định 90 ảnh) cho `cls_threshold` + 4 phương pháp.
3. Eval trên 1,449 ảnh Val, native resolution, dùng chung SAM embedding cho mọi backbone/method.
4. Agent (không phải notebook) tự ghi/cập nhật 2 file local:
   `.mentor/project_changes.md` và `.mentor/agent_knowledge.md`.

## Trước khi chạy
- Trên Colab, notebook clone repo từ GitHub → **push `src/` hỗ trợ ConvNeXt** trước.
- Lần đầu tải VOC2012 (~2 GB), SAM checkpoint (~375 MB với `vit_b`), weight ImageNet ConvNeXt.
- `BATCH_SIZE_BY_BACKBONE`: Tiny = 8, Small = 4 (an toàn GPU 8–16 GB). Giảm nếu OOM.
- Có thể đổi `SAM_MODEL_TYPE = "vit_l"` nếu muốn giống Optuna gốc hơn (chậm/tốn VRAM hơn).

## 1. Cấu hình tổng thể

- `RESUME_IF_EXISTS=True`: nạp lại checkpoint/Optuna bundle đã có, không train/tune lại.
- `TUNE_PROTOCOL_VERSION`: tăng khi đổi miền tune/thuật toán → buộc re-tune.
- LayerCAM dùng 3 stage ConvNeXt: `features.3` (H/8), `features.5` (H/16), `features.7` (H/32).

In [ ]:
# =====================================================================
# 1. CẤU HÌNH
# =====================================================================
BACKBONES = ["convnext_tiny", "convnext_small"]

IMAGE_SIZE = 448
LAYER4_STRIDE_1 = True            # ConvNeXt: last downsample stride 2 -> 1 (14x14 -> 27x27 @448)
TRAIN_SPLIT = "main_trainval"     # "main_trainval" (10,418) hoặc "seg_train" (1,464)

NUM_EPOCHS = 15
BATCH_SIZE_BY_BACKBONE = {"convnext_tiny": 8, "convnext_small": 4}
EVAL_BATCH_SIZE = 8
LR_SCHEDULER = "poly"
POLY_POWER = 0.9
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
NUM_WORKERS = 2
REQUIRE_GPU = True

EXPERIMENT_NAME = "Optuna_ConvNeXt_TinySmall_4Methods"
RESUME_IF_EXISTS = True
RETRAIN = False                   # True: train lại từ đầu, bỏ qua checkpoint cũ
PRETRAINED_RUN_NAME = ""          # Tên run cũ trên Drive nếu muốn dùng lại checkpoint (tuỳ chọn)

# --- CAM / LayerCAM ---
CAM_SCALES = (0.75, 1.0, 1.25)
CAM_HFLIP = True
LAYERCAM_LAYERS = ("features.3", "features.5", "features.7")
LAYERCAM_SHALLOW_GAMMA = 2.0

# --- Ngưỡng mặc định (Optuna sẽ tune lại) ---
CLS_THRESHOLD = 0.5
CAM_THRESHOLD = 0.25
LOW_THRESHOLD = 0.12
BG_THRESHOLD = 0.05
USE_NEG_POINTS = True
BOX_PAD = 24

# --- Optuna ---
CAM_TUNE_MAX_IMAGES = 260
TUNE_SUBSET_SIZE = 90
OPTUNA_CLS_TRIALS = 51
OPTUNA_CAM_TRIALS = 35
OPTUNA_SAM_TRIALS = 60
OPTUNA_LAYERCAM_TRIALS = 35
OPTUNA_LAYERCAM_SAM_TRIALS = 60
TUNE_PROTOCOL_VERSION = 4

# --- SAM ---
SAM_MODEL_TYPE = "vit_b"          # "vit_b" (~375MB), "vit_l" (~1.25GB), "vit_h" (~2.56GB)

# --- Đường dẫn / GitHub ---
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS331_WSSS"
REPO_URL = "https://github.com/minhthanh224/CS331-Project.git"
REPO_BRANCH = "main"

METHODS = ["Vanilla CAM", "CAM + SAM", "LayerCAM", "LayerCAM + SAM"]

assert all(b in {"convnext_tiny", "convnext_small", "convnext_base", "convnext_large"} for b in BACKBONES)
assert IMAGE_SIZE >= 32 and NUM_EPOCHS > 0
assert all(bs > 0 for bs in BATCH_SIZE_BY_BACKBONE.values())
assert TRAIN_SPLIT in {"main_trainval", "seg_train"}
assert LR_SCHEDULER == "poly" and POLY_POWER > 0
assert SAM_MODEL_TYPE in {"vit_b", "vit_l", "vit_h"}
assert all(l.startswith("features.") or l in {"layer2", "layer3", "layer4"} for l in LAYERCAM_LAYERS)
assert 0 < TUNE_SUBSET_SIZE <= CAM_TUNE_MAX_IMAGES
assert TUNE_PROTOCOL_VERSION >= 1

## 2. Môi trường: Colab / VS Code, GPU, cài `segment-anything`, clone repo

- **Colab**: mount Google Drive, cài `segment-anything`, clone/pull repo.
- **VS Code local**: dùng repo hiện tại, cần cài sẵn `segment-anything` và có GPU.

In [ ]:
import gc
import json
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "optuna", "scikit-learn",
         "git+https://github.com/facebookresearch/segment-anything.git"],
        check=True,
    )
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DRIVE_BASE = Path(DRIVE_PROJECT_DIR)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

    PROJECT_ROOT = Path("/content/CS331-Project")
    if not (PROJECT_ROOT / ".git").exists():
        if PROJECT_ROOT.exists():
            shutil.rmtree(PROJECT_ROOT)
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "origin", REPO_BRANCH], check=False)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name in ("notebooks", "test"):
        PROJECT_ROOT = PROJECT_ROOT.parent
    DRIVE_BASE = PROJECT_ROOT

project_root = PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torchvision.datasets.utils import check_integrity, download_url, extract_archive
from tqdm.auto import tqdm

from src import (
    SAM_MODEL_SPECS,
    VOC_CLASSES as CLASSES,
    SegmentationEvaluator,
    VOCClassificationDataset,
    build_balanced_tune_table,
    build_classifier,
    build_optimizer,
    build_optuna_pipeline,
    build_sam_predictor,
    build_transforms,
    build_voc_label_table,
    cache_cams,
    cls_summary_row,
    compute_classification_metrics,
    ensure_sam_checkpoint,
    evaluate_segmentation,
    gen_cam_pseudo_mask,
    gen_sam_pseudo_mask,
    get_device,
    get_sam_embedding,
    image_cell,
    load_classifier,
    load_eval_sample,
    load_eval_samples,
    log_eval_results,
    make_cam_fn,
    make_loader,
    mask_cell,
    plot_grid,
    poly_scheduler,
    predict_probs,
    predict_sample_masks,
    read_split_ids,
    read_wsss_train_ids,
    seg_summary_row,
    set_sam_embedding,
    set_seed,
    train_one_epoch,
)

try:
    import segment_anything  # noqa: F401
except ImportError:
    print("Chưa cài segment-anything. Cài bằng:")
    print("pip install git+https://github.com/facebookresearch/segment-anything.git")

set_seed(SEED)
device = get_device(require_gpu=REQUIRE_GPU)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DRIVE_BASE   : {DRIVE_BASE}")
print(f"DEVICE       : {device} ({gpu_name})")

## 3. Chuẩn bị VOC2012, SAM checkpoint và thư mục kết quả

- Colab: ưu tiên archive/checkpoint có sẵn trên Drive, sau đó mới tải.
- Local: dùng `data/VOC2012` và `checkpoints/` trong repo.

In [ ]:
drive_run_dir = DRIVE_BASE / "runs" / EXPERIMENT_NAME
run_dir = drive_run_dir if IN_COLAB else (project_root / "runs" / EXPERIMENT_NAME)
ckpt_dir = run_dir / "checkpoints"
plots_dir = run_dir / "plots"
optuna_dir = run_dir / "optuna"
for d in (run_dir, ckpt_dir, plots_dir, optuna_dir):
    d.mkdir(parents=True, exist_ok=True)
for b_name in BACKBONES:
    (ckpt_dir / b_name).mkdir(parents=True, exist_ok=True)

archive_name = "VOCtrainval_11-May-2012.tar"
archive_md5 = "6cd6e144f989b92b3379bac3b3de84fd"

if IN_COLAB:
    archive_dir = DRIVE_BASE / "archives"
    archive_dir.mkdir(parents=True, exist_ok=True)

    data_root = Path("/content/data")
    data_root.mkdir(parents=True, exist_ok=True)
    voc_root = data_root / "VOCdevkit" / "VOC2012"
    local_archive = data_root / archive_name
    drive_archive = archive_dir / archive_name

    if not voc_root.is_dir():
        if not check_integrity(local_archive, archive_md5):
            if drive_archive.is_file():
                print("Copy VOC2012 archive từ Google Drive...")
                shutil.copy2(drive_archive, local_archive)
            else:
                urls = [
                    "https://data.pjreddie.com/files/" + archive_name,
                    "https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/" + archive_name,
                ]
                for url in urls:
                    try:
                        print(f"Đang tải VOC2012 từ {url} ...")
                        download_url(url, str(data_root), filename=archive_name, md5=archive_md5)
                        break
                    except Exception as exc:
                        print(f"Tải thất bại: {url} ({exc})")
                else:
                    raise RuntimeError("Không tải được VOC2012. Hãy upload archive lên Google Drive.")

        if not check_integrity(local_archive, archive_md5):
            raise RuntimeError("VOC2012 archive sai MD5.")
        extract_archive(str(local_archive), str(data_root))
        if not drive_archive.exists():
            shutil.copy2(local_archive, drive_archive)

    local_sam_ckpt = ensure_sam_checkpoint(SAM_MODEL_TYPE, data_root, cache_dirs=[archive_dir])
    drive_sam_ckpt = archive_dir / SAM_MODEL_SPECS[SAM_MODEL_TYPE][0]
    if not drive_sam_ckpt.is_file():
        shutil.copy2(local_sam_ckpt, drive_sam_ckpt)
else:
    voc_root = project_root / "data" / "VOC2012"
    local_sam_ckpt = ensure_sam_checkpoint(SAM_MODEL_TYPE, project_root / "checkpoints")

print("VOC2012     :", voc_root)
print("SAM ckpt    :", local_sam_ckpt)
print("Run dir     :", run_dir)

## 4. Chia tập dữ liệu & DataLoader

- Train classifier: `Main/trainval.txt \ Seg/val.txt` (**10,418 ảnh**).
- Tune Optuna: subset cân bằng từ `Segmentation/train.txt` (không trùng Val).
- Eval: `Segmentation/val.txt` (**1,449 ảnh**).

In [ ]:
seg_train_ids = read_split_ids(voc_root, "Segmentation", "train")
val_ids = read_split_ids(voc_root, "Segmentation", "val")
train_ids = read_wsss_train_ids(voc_root, TRAIN_SPLIT)

assert len(seg_train_ids) == 1464 and len(val_ids) == 1449
assert set(train_ids).isdisjoint(val_ids) and set(seg_train_ids).isdisjoint(val_ids)

df_train = build_voc_label_table(voc_root, train_ids, CLASSES)
df_seg_train = build_voc_label_table(voc_root, seg_train_ids, CLASSES)
df_val = build_voc_label_table(voc_root, val_ids, CLASSES)
df_tune_balanced = build_balanced_tune_table(
    df_seg_train, train_ids, classes=CLASSES, max_images=CAM_TUNE_MAX_IMAGES, seed=SEED
)

print(f"1. Train classifier (Main/trainval \\ Seg/val): {len(df_train):5d} ảnh")
print(f"2. Tune đầy đủ (Seg/train)                   : {len(df_seg_train):5d} ảnh")
print(f"   -> Tune cân bằng cho Optuna               : {len(df_tune_balanced):5d} ảnh")
print(f"3. Eval (Seg/val)                            : {len(df_val):5d} ảnh")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
x = np.arange(len(CLASSES))
w = 0.28
axes[0].bar(x - w, df_train[list(CLASSES)].sum(), width=w, label=f"Main/trainval - Seg/val ({len(df_train):,})")
axes[0].bar(x, df_seg_train[list(CLASSES)].sum(), width=w, label=f"Seg/train ({len(df_seg_train):,})")
axes[0].bar(x + w, df_val[list(CLASSES)].sum(), width=w, label=f"Seg/val ({len(df_val):,})")
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASSES, rotation=45, ha="right")
axes[0].set_ylabel("Số lượng ảnh")
axes[0].set_title("Phân bố 20 lớp: Train / Seg-train / Val")
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.35)

axes[1].bar(x, df_tune_balanced[list(CLASSES)].sum(), color="#2b8cbe", width=0.55)
axes[1].set_xticks(x)
axes[1].set_xticklabels(CLASSES, rotation=45, ha="right")
axes[1].set_ylabel("Số lượng ảnh")
axes[1].set_title(f"Tập Tune Cân Bằng cho Optuna ({len(df_tune_balanced)} ảnh)")
axes[1].grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(plots_dir / "class_distribution_splits.png", dpi=160)
plt.show()

train_transform = build_transforms(IMAGE_SIZE, train=True)
eval_transform = build_transforms(IMAGE_SIZE, train=False)

train_dataset = VOCClassificationDataset(voc_root, df_train, CLASSES, train_transform)
seg_train_dataset = VOCClassificationDataset(voc_root, df_seg_train, CLASSES, eval_transform)
tune_balanced_dataset = VOCClassificationDataset(voc_root, df_tune_balanced, CLASSES, eval_transform)
val_dataset = VOCClassificationDataset(voc_root, df_val, CLASSES, eval_transform)

seg_train_loader = make_loader(seg_train_dataset, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device)
val_loader = make_loader(val_dataset, EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, device=device)
criterion = nn.BCEWithLogitsLoss()

print(f"Train dataset: {len(train_dataset):,} | Val dataset: {len(val_dataset):,}")

## 5. Train ConvNeXt-Tiny & ConvNeXt-Small

- ImageNet pretrained → head `classifier[2]` = 20 lớp VOC, fine-tune `BCEWithLogitsLoss`.
- Poly LR theo iteration, `head_lr` gấp 10x `backbone_lr`.
- Chọn `best.pt` theo **Val mAP** (multi-label, chỉ dùng nhãn XML).
- `RESUME_IF_EXISTS=True`: có `best.pt` thì nạp lại; `RETRAIN=True` để train lại.

In [ ]:
config_record = {
    "experiment_name": EXPERIMENT_NAME,
    "backbones": list(BACKBONES),
    "image_size": IMAGE_SIZE,
    "layer4_stride1": LAYER4_STRIDE_1,
    "num_epochs": NUM_EPOCHS,
    "batch_size_by_backbone": dict(BATCH_SIZE_BY_BACKBONE),
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler": LR_SCHEDULER,
    "poly_power": POLY_POWER,
    "train_split": TRAIN_SPLIT,
    "train_num_images": len(train_ids),
    "tune_split": f"Segmentation/train ({len(seg_train_ids)})",
    "tune_balanced_images": len(df_tune_balanced),
    "val_split": f"Segmentation/val ({len(val_ids)})",
    "cam_scales": list(CAM_SCALES),
    "cam_hflip": CAM_HFLIP,
    "layercam_layers": list(LAYERCAM_LAYERS),
    "layercam_shallow_gamma": LAYERCAM_SHALLOW_GAMMA,
    "use_neg_points": USE_NEG_POINTS,
    "sam_model": SAM_MODEL_TYPE,
    "tune_protocol_version": TUNE_PROTOCOL_VERSION,
    "eval_protocol": "original_resolution",
    "seed": SEED,
    "torch_version": str(torch.__version__),
}
(run_dir / "config.json").write_text(json.dumps(config_record, indent=2, ensure_ascii=False), encoding="utf-8")

trained_models = {}
all_histories = {}


def _find_saved_checkpoint(b_name):
    search_dirs = [ckpt_dir / b_name]
    if PRETRAINED_RUN_NAME:
        search_dirs.append(DRIVE_BASE / "runs" / PRETRAINED_RUN_NAME / "checkpoints" / b_name)
    search_dirs.append(project_root / "checkpoints" / b_name)
    for s_dir in search_dirs:
        for fname in ("best.pt", "best_weights_only.pt"):
            cand = s_dir / fname
            if cand.is_file():
                return cand
    return None


def _last_stage_stride(model):
    if hasattr(model, "layer4"):
        block = model.layer4[0]
        return block.conv2.stride if hasattr(block, "conv2") else block.conv1.stride
    return model.features[-2][1].stride


def _history_from_ckpt(ckpt_path):
    try:
        payload = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    except Exception:
        return []
    hist = payload.get("history", []) if isinstance(payload, dict) else []
    last_path = Path(ckpt_path).parent / "last.pt"
    if last_path.is_file():
        try:
            last_payload = torch.load(last_path, map_location="cpu", weights_only=True)
            last_hist = last_payload.get("history", []) if isinstance(last_payload, dict) else []
            if len(last_hist) > len(hist):
                hist = last_hist
        except Exception:
            pass
    return hist


for b_name in BACKBONES:
    b_dir = ckpt_dir / b_name
    b_dir.mkdir(parents=True, exist_ok=True)
    best_weights_path = b_dir / "best_weights_only.pt"
    hist_json_path = b_dir / "training_history.json"
    found_ckpt = None if RETRAIN else _find_saved_checkpoint(b_name)

    if RESUME_IF_EXISTS and found_ckpt is not None:
        stride_arg = LAYER4_STRIDE_1 if found_ckpt.name == "best_weights_only.pt" else None
        trained_models[b_name] = load_classifier(
            b_name, found_ckpt, device=device, num_classes=len(CLASSES),
            layer4_stride1=stride_arg,
        )
        all_histories[b_name] = _history_from_ckpt(found_ckpt)
        stride = _last_stage_stride(trained_models[b_name])
        print(
            f"[Resume] {b_name}: dùng {found_ckpt} | stride={stride} | "
            f"{len(all_histories[b_name])}/{NUM_EPOCHS} epochs"
        )
        if len(all_histories[b_name]) < NUM_EPOCHS:
            print("  [Cảnh báo] Checkpoint chưa đủ epoch; đặt RETRAIN=True nếu muốn train lại.")
        continue

    batch_size = BATCH_SIZE_BY_BACKBONE[b_name]
    train_loader = make_loader(
        train_dataset, batch_size, shuffle=True,
        num_workers=NUM_WORKERS, device=device, seed=SEED,
    )
    model = build_classifier(
        model_name=b_name, num_classes=len(CLASSES), layer4_stride1=LAYER4_STRIDE_1
    ).to(device)
    optimizer = build_optimizer(
        model, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = poly_scheduler(
        optimizer, total_steps=NUM_EPOCHS * len(train_loader), power=POLY_POWER
    )

    history = []
    best_val_map = -1.0
    best_epoch = 0

    print("=" * 80)
    print(
        f"TRAIN {b_name} | {NUM_EPOCHS} epochs | bs={batch_size} | "
        f"stride={_last_stage_stride(model)} | {len(train_dataset):,} ảnh"
    )
    print("=" * 80)

    for epoch in range(1, NUM_EPOCHS + 1):
        current_lr = optimizer.param_groups[0]["lr"]
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, device,
            scheduler=scheduler,
            desc=f"[{b_name}] Epoch {epoch:02d}/{NUM_EPOCHS:02d} [Train]",
        )
        val_probs, val_targets, val_loss = predict_probs(
            model, val_loader, device, criterion=criterion,
            desc=f"[{b_name}] Epoch {epoch:02d} [Val Cls]", leave=False,
        )
        val_cls = compute_classification_metrics(
            val_targets, val_probs, threshold=CLS_THRESHOLD
        )
        val_map = val_cls["mAP"]
        val_macro_f1 = val_cls["macro_f1"]

        history.append({
            "backbone": b_name,
            "epoch": epoch,
            "backbone_lr": current_lr,
            "train_loss": round(train_loss, 6),
            "val_loss": round(val_loss, 6),
            "val_mAP": round(val_map, 6),
            "val_macro_f1": round(val_macro_f1, 6),
        })

        improved = val_map > best_val_map
        if improved:
            best_val_map = val_map
            best_epoch = epoch

        ckpt_payload = {
            "backbone": b_name,
            "epoch_in_run": epoch,
            "best_epoch": best_epoch,
            "classes": list(CLASSES),
            "model_state_dict": model.state_dict(),
            "val_loss": val_loss,
            "val_mAP": val_map,
            "best_val_mAP": best_val_map,
            "history": list(history),
            "config": config_record,
        }
        torch.save(ckpt_payload, b_dir / "last.pt")
        if improved:
            torch.save(ckpt_payload, b_dir / "best.pt")
            torch.save(model.state_dict(), best_weights_path)

        hist_json_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
        pd.DataFrame(history).to_csv(b_dir / "training_history.csv", index=False)

        star = " [BEST]" if improved else ""
        print(
            f"[{b_name}] Epoch {epoch:02d}/{NUM_EPOCHS:02d} | LR: {current_lr:.2e} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Val mAP: {val_map:.4f} | Macro F1: {val_macro_f1:.4f}{star}"
        )

    trained_models[b_name] = load_classifier(
        b_name, best_weights_path, device=device, num_classes=len(CLASSES),
        layer4_stride1=LAYER4_STRIDE_1,
    )
    all_histories[b_name] = history
    del model, optimizer, scheduler, train_loader
    gc.collect()
    torch.cuda.empty_cache()

# Bảng tóm tắt checkpoint
summary_rows = []
for b_name, hist in all_histories.items():
    if not hist:
        continue
    best = max(hist, key=lambda row: row.get("val_mAP", -1))
    summary_rows.append({
        "Backbone": b_name,
        "Epochs saved": len(hist),
        "Best epoch": best.get("epoch"),
        "Best Val mAP": best.get("val_mAP"),
        "Best Val Macro F1": best.get("val_macro_f1"),
    })
if summary_rows:
    display(pd.DataFrame(summary_rows))

# Đường cong huấn luyện
if any(all_histories.values()):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    colors = {"convnext_tiny": "#1f77b4", "convnext_small": "#ff7f0e"}
    for b_name, hist in all_histories.items():
        if not hist:
            continue
        color = colors.get(b_name)
        epochs = [row["epoch"] for row in hist]
        axes[0].plot(epochs, [row["train_loss"] for row in hist], "--", alpha=0.65, color=color)
        axes[0].plot(epochs, [row["val_loss"] for row in hist], marker="o", linewidth=2, color=color, label=b_name)
        axes[1].plot(epochs, [row["val_mAP"] * 100 for row in hist], marker="o", linewidth=2, color=color, label=b_name)
        axes[2].plot(epochs, [row["val_macro_f1"] * 100 for row in hist], marker="s", linewidth=2, color=color, label=b_name)

    axes[0].set_title("Train (dashed) & Val BCE Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[1].set_title("Val Classification mAP (%)")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP (%)")
    axes[2].set_title("Val Macro F1 (%)")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Macro F1 (%)")
    for ax in axes:
        ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(plots_dir / "all_backbones_training_curves.png", dpi=160, bbox_inches="tight")
    plt.show()

## 6. Khởi tạo SAM & cache embedding trên tập tune

- Mỗi ảnh tune chỉ encode **1 lần**; embedding lưu trên CPU và dùng lại cho mọi trial/backbone/nhánh SAM.
- Đây là lý do chính giúp tune 2 backbone × 2 nhánh SAM khả thi trong một phiên Colab.

In [ ]:
predictor = build_sam_predictor(SAM_MODEL_TYPE, local_sam_ckpt, device=device)
print(f"SamPredictor ({SAM_MODEL_TYPE}) đã sẵn sàng trên {device}")

shared_tune_images = load_eval_samples(
    tune_balanced_dataset,
    image_size=None,
    desc=f"Loading Tune Images & Native GT Masks ({len(tune_balanced_dataset)} ảnh)",
)

tune_step = max(1, len(shared_tune_images) // TUNE_SUBSET_SIZE)
tune_indices = list(range(0, len(shared_tune_images), tune_step))[:TUNE_SUBSET_SIZE]
tune_samples = [shared_tune_images[i] for i in tune_indices]

shared_sam_embeds = {}
for local_i, sample in enumerate(tqdm(
    tune_samples, desc=f"Caching Shared SAM ({SAM_MODEL_TYPE}) Embeddings ({len(tune_samples)} ảnh)"
)):
    predictor.set_image(sample["img_u8"])
    shared_sam_embeds[local_i] = get_sam_embedding(predictor)

print(
    f"Tune samples: {len(tune_samples)} | SAM embeddings: {len(shared_sam_embeds)} | "
    f"Toàn bộ tune set: {len(shared_tune_images)}"
)

## 7. Optuna cho 4 phương pháp (không DenseCRF)

Với từng backbone, tune trên **cùng một subset `TUNE_SUBSET_SIZE` ảnh** để kết quả so sánh được:

1. `cls_threshold` (0.25–0.75) theo Macro-F1 trên `Seg/train`.
2. **Vanilla CAM**: `cam_threshold` (0.06–0.40).
3. **CAM + SAM**: `cam_threshold`, `low_ratio`, `bg_ratio`, `box_pad`, 4 trọng số score, `cam_pixel_weight`.
4. **LayerCAM**: `cam_threshold` (0.06–0.40).
5. **LayerCAM + SAM**: cùng bộ tham số như CAM + SAM.

Bundle lưu `tune_protocol_version`, `layer4_stride1`, `layercam_layers`, `cam_scales`, `cam_hflip`,
`sam_model`, `use_neg_points` để tránh resume nhầm cấu hình cũ.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

all_best_params = {}
id_to_seg_idx = {img_id: i for i, img_id in enumerate(seg_train_dataset.image_ids)}
required_optuna_keys = (
    "tune_protocol_version",
    "best_cls_threshold",
    "best_cam_params",
    "best_sam_params",
    "best_layercam_params",
    "best_layercam_sam_params",
)


def _normalize_sam_score_weights(w_cam_iou, w_core_recall, w_sam_iou, w_bg_spill):
    total = max(float(w_cam_iou + w_core_recall + w_sam_iou + w_bg_spill), 1e-8)
    return {
        "w_cam_iou": round(float(w_cam_iou) / total, 5),
        "w_core_recall": round(float(w_core_recall) / total, 5),
        "w_sam_iou": round(float(w_sam_iou) / total, 5),
        "w_bg_spill": round(float(w_bg_spill) / total, 5),
    }


def _sam_params_for_eval(params):
    cam_threshold = float(params["cam_threshold"])
    low_threshold = round(float(params["low_ratio"]) * cam_threshold, 4)
    bg_threshold = (
        round(float(params.get("bg_ratio", 0.0)) * low_threshold, 4) if USE_NEG_POINTS else 0.0
    )
    return {
        "cam_threshold": cam_threshold,
        "low_threshold": low_threshold,
        "bg_threshold": bg_threshold,
        "box_pad": int(params["box_pad"]),
        "cam_pixel_weight": float(params["cam_pixel_weight"]),
        "use_neg_points": USE_NEG_POINTS,
        **_normalize_sam_score_weights(
            w_cam_iou=params["w_cam_iou"],
            w_core_recall=params["w_core_recall"],
            w_sam_iou=params["w_sam_iou"],
            w_bg_spill=params["w_bg_spill"],
        ),
    }


for b_name in BACKBONES:
    model = trained_models[b_name]
    b_param_file = optuna_dir / f"{b_name}_best_params.json"

    if RESUME_IF_EXISTS and b_param_file.is_file():
        loaded_params = json.loads(b_param_file.read_text(encoding="utf-8"))
        if (
            all(key in loaded_params for key in required_optuna_keys)
            and loaded_params.get("sam_model") == SAM_MODEL_TYPE
            and loaded_params.get("eval_protocol") == "original_resolution"
            and loaded_params.get("use_neg_points") == USE_NEG_POINTS
            and loaded_params.get("tune_protocol_version") == TUNE_PROTOCOL_VERSION
            and loaded_params.get("layer4_stride1") == LAYER4_STRIDE_1
            and list(loaded_params.get("layercam_layers", [])) == list(LAYERCAM_LAYERS)
            and list(loaded_params.get("cam_scales", [])) == list(CAM_SCALES)
            and loaded_params.get("cam_hflip") == CAM_HFLIP
        ):
            print(f"[Resume] Nạp Optuna bundle của '{b_name}' từ {b_param_file}")
            all_best_params[b_name] = loaded_params
            continue
        print(f"[Re-tune] Bundle '{b_name}' không khớp cấu hình hiện tại -> tune lại.")

    print("\n" + "=" * 80)
    print(f"OPTUNA 4 PHƯƠNG PHÁP CHO BACKBONE: {b_name.upper()}")
    print("=" * 80)

    seg_probs_all, seg_targets_all, _ = predict_probs(
        model, seg_train_loader, device, desc=f"[{b_name}] Infer Seg/Train Probs"
    )

    def obj_cls(trial):
        threshold = trial.suggest_float("cls_threshold", 0.25, 0.75, step=0.01)
        return compute_classification_metrics(
            seg_targets_all, seg_probs_all, threshold=threshold
        )["macro_f1"]

    study_cls = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study_cls.optimize(obj_cls, n_trials=OPTUNA_CLS_TRIALS)
    best_cls_thr = study_cls.best_params["cls_threshold"]
    study_cls.trials_dataframe().to_csv(optuna_dir / f"{b_name}_trials_cls.csv", index=False)
    print(f"[{b_name}] Best cls_threshold = {best_cls_thr:.2f} | Macro-F1 = {study_cls.best_value:.4f}")

    tune_probs = np.stack([seg_probs_all[id_to_seg_idx[item["img_id"]]] for item in tune_samples])
    cam_caches = cache_cams(
        tune_samples,
        tune_probs,
        cls_threshold=best_cls_thr,
        cam_fns={
            "cam": make_cam_fn(model, method="cam", scales=CAM_SCALES, hflip=CAM_HFLIP),
            "layercam": make_cam_fn(
                model, method="layercam", scales=CAM_SCALES, hflip=CAM_HFLIP,
                layers=LAYERCAM_LAYERS, shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
            ),
        },
        desc=f"[{b_name}] Caching Multi-Scale CAM & LayerCAM",
    )
    b_cams_cache = cam_caches["cam"]
    b_layercams_cache = cam_caches["layercam"]

    def _run_cam_study(cache_list, n_trials, csv_name, label):
        def _obj(trial):
            cam_thr = trial.suggest_float("cam_threshold", 0.06, 0.40, step=0.01)
            evaluator = SegmentationEvaluator(num_classes=21, ignore_index=255)
            for base_item, cam_item in zip(tune_samples, cache_list):
                pseudo_mask = gen_cam_pseudo_mask(
                    cam_item["cams"], cam_item["active_classes"], cam_threshold=cam_thr
                )
                evaluator.update(pseudo_mask, base_item["gt_mask"])
            return evaluator.compute()["mIoU"]

        study = optuna.create_study(
            direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        study.optimize(_obj, n_trials=n_trials, show_progress_bar=True)
        study.trials_dataframe().to_csv(optuna_dir / csv_name, index=False)
        print(f"[{b_name} | {label}] Best: {study.best_params} | Tune mIoU = {study.best_value:.4f}")
        return study

    def _run_sam_study(cache_list, n_trials, csv_name, label):
        def _obj(trial):
            cam_threshold = trial.suggest_float("cam_threshold", 0.10, 0.36, step=0.02)
            low_ratio = trial.suggest_float("low_ratio", 0.35, 0.90, step=0.05)
            low_threshold = round(low_ratio * cam_threshold, 4)
            bg_ratio = (
                trial.suggest_float("bg_ratio", 0.05, 0.60, step=0.05) if USE_NEG_POINTS else 0.0
            )
            bg_threshold = round(bg_ratio * low_threshold, 4)
            box_pad = trial.suggest_int("box_pad", 8, 64, step=4)
            w_cam_iou = trial.suggest_float("w_cam_iou", 0.20, 0.70, step=0.05)
            w_core_recall = trial.suggest_float("w_core_recall", 0.05, 0.55, step=0.05)
            w_sam_iou = trial.suggest_float("w_sam_iou", 0.10, 0.40, step=0.05)
            w_bg_spill = trial.suggest_float("w_bg_spill", 0.20, 0.90, step=0.05)
            cam_pixel_weight = trial.suggest_float("cam_pixel_weight", 0.35, 0.90, step=0.05)
            score_weights = _normalize_sam_score_weights(
                w_cam_iou=w_cam_iou, w_core_recall=w_core_recall,
                w_sam_iou=w_sam_iou, w_bg_spill=w_bg_spill,
            )

            evaluator = SegmentationEvaluator(num_classes=21, ignore_index=255)
            for local_i, base_item in enumerate(tune_samples):
                cam_item = cache_list[local_i]
                set_sam_embedding(predictor, shared_sam_embeds[local_i], device=device)
                pseudo_mask = gen_sam_pseudo_mask(
                    predictor=predictor,
                    image_rgb=base_item["img_rgb"],
                    cams=cam_item["cams"],
                    class_indices=cam_item["active_classes"],
                    cam_threshold=cam_threshold,
                    low_threshold=low_threshold,
                    bg_threshold=bg_threshold,
                    use_neg_points=USE_NEG_POINTS,
                    box_pad=box_pad,
                    encode_image=False,
                    w_cam_iou=score_weights["w_cam_iou"],
                    w_core_recall=score_weights["w_core_recall"],
                    w_sam_iou=score_weights["w_sam_iou"],
                    w_bg_spill=score_weights["w_bg_spill"],
                    cam_pixel_weight=cam_pixel_weight,
                )
                evaluator.update(pseudo_mask, base_item["gt_mask"])
            return evaluator.compute()["mIoU"]

        study = optuna.create_study(
            direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        study.optimize(_obj, n_trials=n_trials, show_progress_bar=True)
        study.trials_dataframe().to_csv(optuna_dir / csv_name, index=False)
        print(f"[{b_name} | {label}] Best: {study.best_params} | Tune mIoU = {study.best_value:.4f}")
        return study

    study_cam = _run_cam_study(
        b_cams_cache, OPTUNA_CAM_TRIALS, f"{b_name}_trials_cam.csv", "Vanilla CAM"
    )
    study_sam = _run_sam_study(
        b_cams_cache, OPTUNA_SAM_TRIALS, f"{b_name}_trials_sam.csv", f"CAM + SAM ({SAM_MODEL_TYPE})"
    )
    study_layercam = _run_cam_study(
        b_layercams_cache, OPTUNA_LAYERCAM_TRIALS, f"{b_name}_trials_layercam.csv", "LayerCAM"
    )
    study_layercam_sam = _run_sam_study(
        b_layercams_cache, OPTUNA_LAYERCAM_SAM_TRIALS,
        f"{b_name}_trials_layercam_sam.csv", f"LayerCAM + SAM ({SAM_MODEL_TYPE})",
    )

    bundle = {
        "backbone": b_name,
        "image_size": IMAGE_SIZE,
        "tune_protocol_version": TUNE_PROTOCOL_VERSION,
        "tune_subset_size": len(tune_samples),
        "eval_protocol": "original_resolution",
        "use_neg_points": USE_NEG_POINTS,
        "layer4_stride1": LAYER4_STRIDE_1,
        "sam_model": SAM_MODEL_TYPE,
        "cam_scales": list(CAM_SCALES),
        "cam_hflip": CAM_HFLIP,
        "layercam_layers": list(LAYERCAM_LAYERS),
        "layercam_shallow_gamma": LAYERCAM_SHALLOW_GAMMA,
        "best_cls_threshold": best_cls_thr,
        "best_cam_params": study_cam.best_params,
        "best_sam_params": _sam_params_for_eval(study_sam.best_params),
        "best_layercam_params": study_layercam.best_params,
        "best_layercam_sam_params": _sam_params_for_eval(study_layercam_sam.best_params),
        "tune_best_miou": {
            "vanilla_cam": round(float(study_cam.best_value), 4),
            "cam_sam": round(float(study_sam.best_value), 4),
            "layercam": round(float(study_layercam.best_value), 4),
            "layercam_sam": round(float(study_layercam_sam.best_value), 4),
        },
    }
    b_param_file.write_text(json.dumps(bundle, indent=2, ensure_ascii=False), encoding="utf-8")
    all_best_params[b_name] = bundle

    del b_cams_cache, b_layercams_cache
    gc.collect()
    torch.cuda.empty_cache()

(optuna_dir / "all_backbones_best_params.json").write_text(
    json.dumps(all_best_params, indent=2, ensure_ascii=False), encoding="utf-8"
)

shared_sam_embeds.clear()
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------------
# Biểu đồ hội tụ Optuna + so sánh best tune mIoU
# ---------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 9.2))
colors = {"convnext_tiny": "#1f77b4", "convnext_small": "#ff7f0e"}
specs = [
    (axes[0, 0], "trials_cam.csv", True, "Vanilla CAM: threshold vs mIoU"),
    (axes[0, 1], "trials_sam.csv", False, f"CAM + SAM ({SAM_MODEL_TYPE}) convergence"),
    (axes[1, 0], "trials_layercam.csv", True, "LayerCAM: threshold vs mIoU"),
    (axes[1, 1], "trials_layercam_sam.csv", False, f"LayerCAM + SAM ({SAM_MODEL_TYPE}) convergence"),
]
for ax, csv_suffix, is_threshold, title in specs:
    for b_name in BACKBONES:
        csv_path = optuna_dir / f"{b_name}_{csv_suffix}"
        if not csv_path.is_file():
            continue
        df_trials = pd.read_csv(csv_path)
        color = colors.get(b_name)
        if is_threshold:
            df_trials = df_trials.sort_values("params_cam_threshold")
            ax.plot(
                df_trials["params_cam_threshold"], df_trials["value"] * 100,
                marker="o", color=color, label=b_name,
            )
        else:
            ax.plot(
                df_trials["number"], df_trials["value"].cummax() * 100,
                marker=".", linewidth=2, color=color, label=b_name,
            )
    ax.set_title(title)
    ax.set_xlabel("cam_threshold" if is_threshold else "Trial")
    ax.set_ylabel("Tune mIoU (%)")
    ax.grid(alpha=0.3)
    ax.legend()

tune_keys = ["vanilla_cam", "cam_sam", "layercam", "layercam_sam"]
tune_labels = ["CAM", "CAM+SAM", "LayerCAM", "LCAM+SAM"]
for idx_b, b_name in enumerate(BACKBONES):
    values = [all_best_params[b_name]["tune_best_miou"][key] * 100 for key in tune_keys]
    pos = np.arange(len(tune_keys)) + (idx_b - (len(BACKBONES) - 1) / 2) * 0.28
    axes[0, 2].bar(pos, values, width=0.28, color=colors.get(b_name), label=b_name)
axes[0, 2].set_xticks(np.arange(len(tune_keys)))
axes[0, 2].set_xticklabels(tune_labels, rotation=15)
axes[0, 2].set_title("Best Tune mIoU (%) — 4 methods")
axes[0, 2].set_ylabel("Tune mIoU (%)")
axes[0, 2].grid(axis="y", alpha=0.3)
axes[0, 2].legend()

axes[1, 2].axis("off")
axes[1, 2].text(
    0.0, 0.9,
    "Ghi chú:\n"
    "- DenseCRF đã bị loại bỏ.\n"
    "- Tune trên subset cân bằng từ Seg/train.\n"
    "- Eval cuối trên 1,449 ảnh Seg/val.\n"
    "- Bundle lưu tại optuna/*.json",
    fontsize=11, va="top",
)
plt.tight_layout()
plt.savefig(plots_dir / "optuna_history.png", dpi=160)
plt.show()

# ---------------------------------------------------------------
# Hyperparameter importance (chỉ 2 nhánh SAM)
# ---------------------------------------------------------------
from sklearn.ensemble import RandomForestRegressor


def compute_csv_param_importance(csv_path, seed=42):
    df = pd.read_csv(csv_path).dropna(subset=["value"])
    param_cols = [c for c in df.columns if c.startswith("params_")]
    if len(df) < 4 or not param_cols:
        return pd.Series(dtype=float)
    X = df[param_cols].copy()
    y = df["value"].values
    valid_cols = [c for c in param_cols if X[c].nunique() > 1]
    if not valid_cols:
        return pd.Series(1.0 / len(param_cols), index=[c.replace("params_", "") for c in param_cols])
    rf = RandomForestRegressor(n_estimators=128, max_depth=6, random_state=seed)
    rf.fit(X[valid_cols], y)
    imp = pd.Series(rf.feature_importances_, index=[c.replace("params_", "") for c in valid_cols])
    return (imp / imp.sum()) if imp.sum() > 0 else imp


fig_imp, axes_imp = plt.subplots(2, len(BACKBONES), figsize=(6.2 * len(BACKBONES), 8.4), squeeze=False)
importance_records = []
method_imp_specs = [
    ("CAM + SAM", "trials_sam.csv", "#e6550d"),
    ("LayerCAM + SAM", "trials_layercam_sam.csv", "#c44e52"),
]
for row_idx, (method_title, csv_suffix, bar_color) in enumerate(method_imp_specs):
    for col_idx, b_name in enumerate(BACKBONES):
        ax = axes_imp[row_idx, col_idx]
        csv_file = optuna_dir / f"{b_name}_{csv_suffix}"
        if csv_file.is_file():
            imp_series = compute_csv_param_importance(csv_file, seed=SEED).sort_values(ascending=True)
            bars = ax.barh(
                imp_series.index, imp_series.values * 100,
                color=bar_color, alpha=0.88, edgecolor="black", height=0.6,
            )
            for bar in bars:
                width = bar.get_width()
                ax.text(
                    width + 1.0, bar.get_y() + bar.get_height() / 2,
                    f"{width:.1f}%", va="center", fontsize=8.5, fontweight="bold",
                )
            ax.set_xlim(0, max(100.0, float(imp_series.max() * 100) + 18.0))
            for p_name, p_val in imp_series.sort_values(ascending=False).items():
                importance_records.append({
                    "Method": method_title,
                    "Backbone": b_name,
                    "Hyperparameter": p_name,
                    "Importance (%)": round(float(p_val * 100), 2),
                })
        ax.set_title(f"{method_title} — {b_name}", fontsize=11, fontweight="bold")
        ax.set_xlabel("Mức độ ảnh hưởng (%)")
        ax.grid(axis="x", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.savefig(plots_dir / "optuna_param_importance.png", dpi=160)
plt.show()

if importance_records:
    df_imp = pd.DataFrame(importance_records).sort_values(
        ["Method", "Backbone", "Importance (%)"], ascending=[True, True, False]
    )
    df_imp.to_csv(optuna_dir / "param_importance_summary.csv", index=False)
    display(df_imp)

## 8. Đánh giá cuối: 2 backbone × 4 phương pháp = 8 cấu hình

**Tăng tốc**: `predictor.set_image` chạy **1 lần/ảnh**, dùng chung cho:
- 2 backbone,
- `CAM + SAM` và `LayerCAM + SAM`.

Kết quả lưu:
- `val_classification_summary.csv`
- `val_segmentation_8configs_summary.csv`
- `val_per_class_iou.csv`

In [ ]:
val_probs_by_backbone = {}
cls_summary_rows = []
val_targets_all = None

for b_name in BACKBONES:
    cls_thr = all_best_params[b_name]["best_cls_threshold"]
    probs_arr, targets_arr, _ = predict_probs(
        trained_models[b_name], val_loader, device,
        desc=f"Val Classification (1,449 ảnh) [{b_name}]",
    )
    if val_targets_all is None:
        val_targets_all = targets_arr
    val_probs_by_backbone[b_name] = probs_arr

    cls_metrics = compute_classification_metrics(
        val_targets_all, probs_arr, threshold=cls_thr
    )
    log_eval_results(
        experiment_name=f"{b_name}_ConvNeXt_Classification",
        metrics=cls_metrics,
        config={**config_record, "backbone": b_name, "cls_threshold": cls_thr},
        log_dir=run_dir,
    )
    cls_summary_rows.append(
        cls_summary_row(cls_metrics, percent=True, Backbone=b_name, **{"Best Cls Thr": cls_thr})
    )

cls_summary_df = pd.DataFrame(cls_summary_rows)
cls_summary_df.to_csv(run_dir / "val_classification_summary.csv", index=False)
print("=== KẾT QUẢ PHÂN LOẠI ĐA NHÃN TRÊN VAL (1,449 ẢNH) ===")
display(cls_summary_df)

# --- Segmentation: build pipeline từ bundle Optuna (chỉ 4 phương pháp) ---
pipelines = {
    b_name: build_optuna_pipeline(
        model=trained_models[b_name],
        probs=val_probs_by_backbone[b_name],
        best_params=all_best_params[b_name],
        predictor=predictor,
        encode_image=False,
        scales=CAM_SCALES,
        hflip=CAM_HFLIP,
        layercam_layers=LAYERCAM_LAYERS,
        shallow_gamma=LAYERCAM_SHALLOW_GAMMA,
    )
    for b_name in BACKBONES
}

computed_seg_metrics = evaluate_segmentation(
    val_dataset,
    pipelines,
    image_size=None,
    prepare_fn=predictor.set_image,
    desc="Val Segmentation (1,449 ảnh — 2 backbones × 4 methods, shared SAM)",
)

seg_summary_rows = []
per_class_records = {"Class": ["background", *CLASSES]}
for b_name in BACKBONES:
    for method_label in METHODS:
        metrics = computed_seg_metrics[b_name][method_label]
        safe_method = method_label.replace(" ", "").replace("+", "_")
        log_eval_results(
            experiment_name=f"{b_name}_{safe_method}_Optuna_ConvNeXt",
            metrics=metrics,
            config={**all_best_params[b_name], "method": method_label},
            log_dir=run_dir,
        )
        seg_summary_rows.append(
            seg_summary_row(metrics, percent=True, Backbone=b_name, Method=method_label)
        )
        per_class_records[f"{b_name} | {method_label}"] = [
            round(value * 100, 2) for value in metrics["iou_per_class"]
        ]

seg_df = pd.DataFrame(seg_summary_rows)
seg_df.to_csv(run_dir / "val_segmentation_8configs_summary.csv", index=False)
pd.DataFrame(per_class_records).to_csv(run_dir / "val_per_class_iou.csv", index=False)

print("\n=== BẢNG TỔNG HỢP 8 CẤU HÌNH WSSS TRÊN VAL (1,449 ẢNH) ===")
display(seg_df)

# Biểu đồ mIoU
fig, ax = plt.subplots(figsize=(12, 5.6))
x = np.arange(len(BACKBONES))
width = 0.2
method_colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]
for idx, method_label in enumerate(METHODS):
    values = [computed_seg_metrics[b][method_label]["mIoU"] * 100 for b in BACKBONES]
    bars = ax.bar(
        x + (idx - (len(METHODS) - 1) / 2) * width,
        values, width=width, label=method_label, color=method_colors[idx],
    )
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2, height + 0.5,
            f"{height:.1f}%", ha="center", va="bottom", fontsize=8, fontweight="bold",
        )

ax.set_xticks(x)
ax.set_xticklabels([b.upper() for b in BACKBONES], fontsize=11, fontweight="bold")
ax.set_ylabel("Val mIoU (%)")
ax.set_title(
    f"So sánh mIoU trên Val (1,449 ảnh): 2 ConvNeXt × 4 phương pháp "
    f"(SAM {SAM_MODEL_TYPE}, không DenseCRF)"
)
ax.set_ylim(0, seg_df["mIoU (%)"].max() + 8)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.10))
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(plots_dir / "val_miou_8configs.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Trực quan hoá kết quả phân đoạn

Mỗi backbone: `Input | Ground Truth | CAM | LayerCAM | CAM+SAM | LayerCAM+SAM`.

In [ ]:
sample_indices = [0, 12, 25, 42]

for b_name in BACKBONES:
    pipe = pipelines[b_name]
    rows = []
    for idx in sample_indices:
        sample = load_eval_sample(val_dataset, idx, image_size=None)
        predictor.set_image(sample["img_u8"])
        out = predict_sample_masks(sample, pipe, pipe.probs[idx])
        active_names = [CLASSES[c] for c in out["active_classes"]]
        rows.append([
            image_cell(sample["img_rgb"], f"{sample['img_id']}\n{active_names}"),
            mask_cell(sample["gt_mask"], "Ground Truth"),
            mask_cell(out["masks"]["Vanilla CAM"], "Vanilla CAM"),
            mask_cell(out["masks"]["LayerCAM"], "LayerCAM"),
            mask_cell(out["masks"]["CAM + SAM"], "CAM + SAM"),
            mask_cell(out["masks"]["LayerCAM + SAM"], "LayerCAM + SAM"),
        ])
    plot_grid(
        rows, cell_size=3.1,
        save_path=plots_dir / f"val_qualitative_{b_name}.png",
    )

## 10. Tổng kết & danh sách file

In [ ]:
print("=" * 80)
print(f"HOÀN TẤT — kết quả tại: {run_dir}")
print("=" * 80)

if "seg_df" in globals() and len(seg_df) > 0:
    best_row = seg_df.sort_values("mIoU (%)", ascending=False).iloc[0]
    print(
        f"Best mIoU: {best_row['mIoU (%)']:.2f}% | "
        f"{best_row['Backbone']} | {best_row['Method']}"
    )

for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        print(f" - {path.relative_to(run_dir)} ({path.stat().st_size / 1024:,.1f} KB)")